<a href="https://colab.research.google.com/github/usharani-192312304/ITA0605-ML/blob/main/Exp19_Bank_Loan_Naive_Bayes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Experiment 19: Bank Loan Prediction using Naive Bayes
**Course:** ITA06 - Machine Learning (Lab)

## Aim
Implement Naive Bayes classification for Bank Loan prediction.

## Background
Banks decide whether to **approve or reject** a loan from applicant details. Our data mixes numeric (income, credit score, loan amount) and categorical features (education, employment, property area), so we use two Naive Bayes variants: **Gaussian NB** (numeric features modelled as normal distributions, categoricals one-hot encoded) and **Categorical NB** (all features discretised into categories, with **Laplace smoothing** `alpha` to avoid zero probabilities).

## Pipeline followed in this notebook
1. Step 1: Data Collection
2. Step 2: Data Loading
3. Step 3: Data Preprocessing
4. Step 4: Exploratory Data Analysis (EDA)
5. Step 5: Model Training
6. Step 6: Model Evaluation
7. Step 7: Prediction on New Data

> **How to run:** Open in Google Colab -> *Runtime -> Run all*. No files need to be uploaded; the data is created or loaded inside the notebook.

## Setup: Import libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)
np.random.seed(42)
print("Libraries imported successfully")

## Step 1: Data Collection
We generate **1,500 loan applications** resembling the well-known *Loan Prediction* dataset. Approval depends on credit score, income vs loan size, education, marital status, employment and property area (plus noise). Realistic missing values are injected. Saved as `bank_loan.csv`.

In [ ]:
rng = np.random.RandomState(42)
n = 1500
zs = lambda v: (v - v.mean()) / v.std()

age = rng.randint(21, 66, n)
income = np.clip(rng.lognormal(10.2, 0.55, n), 8000, 250000).round(0)               # monthly income
loan_amount = np.clip(income * 12 * rng.uniform(0.5, 4.0, n) / 1000, 50, 2000).round(0)   # in thousands (0.5x - 4x annual income)
loan_term = rng.choice([120, 180, 240, 300, 360], n, p=[.05, .1, .1, .1, .65])
credit_score = np.clip(rng.normal(680, 70, n), 400, 850).round(0)
employment = rng.choice(["Salaried", "Self-employed", "Business", "Unemployed"], n, p=[.55, .2, .2, .05])
education = rng.choice(["Graduate", "Not Graduate"], n, p=[.75, .25])
married = rng.choice(["Yes", "No"], n, p=[.65, .35])
dependents = rng.choice([0, 1, 2, 3], n, p=[.5, .2, .2, .1])
area = rng.choice(["Urban", "Semiurban", "Rural"], n, p=[.35, .4, .25])
existing_emi = (income * rng.uniform(0, 0.3, n)).round(0)                           # monthly EMI = up to 30% of income

z = (0.9 * zs(credit_score) + 0.5 * zs(np.log(income)) - 0.5 * zs(loan_amount * 1000 / (income * 12))
     + 0.25 * (education == "Graduate") + 0.2 * (married == "Yes") - 0.8 * (employment == "Unemployed")
     + 0.15 * (area == "Semiurban") - 0.1 * dependents + rng.normal(0, 0.3, n))
p_approve = 1 / (1 + np.exp(-(0.7 + 2.5 * zs(z))))
status = np.where(rng.rand(n) < p_approve, "Approved", "Rejected")

loan = pd.DataFrame({"Age": age, "Monthly_Income": income, "Loan_Amount_K": loan_amount, "Loan_Term_Months": loan_term,
                     "Credit_Score": credit_score, "Employment": employment, "Education": education, "Married": married,
                     "Dependents": dependents.astype(float), "Property_Area": area, "Existing_EMI": existing_emi, "Loan_Status": status})
for col, frac in {"Credit_Score": 0.05, "Loan_Amount_K": 0.03, "Dependents": 0.02, "Married": 0.01, "Employment": 0.02}.items():
    loan.loc[rng.rand(n) < frac, col] = np.nan
loan.to_csv("bank_loan.csv", index=False)
print("Saved bank_loan.csv", loan.shape)

## Step 2: Data Loading

In [ ]:
df = pd.read_csv("bank_loan.csv")
df.head()

## Step 3: Data Preprocessing
Split the data, then build **two preprocessing pipelines**: (A) for Gaussian NB - impute, scale numerics, one-hot encode categoricals; (B) for Categorical NB - impute, discretise numerics into 5 quantile bins, ordinal-encode categoricals.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder, KBinsDiscretizer, FunctionTransformer

print("Missing values:\n", df.isnull().sum()[df.isnull().sum() > 0])
y = (df["Loan_Status"] == "Approved").astype(int)
X = df.drop(columns="Loan_Status")
num_cols = X.select_dtypes(include="number").columns.tolist()
cat_cols = X.select_dtypes(exclude="number").columns.tolist()
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

def to_int(a):
    return np.asarray(a).astype(int)

# (A) Gaussian NB
prep_gauss = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), num_cols),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")), ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), cat_cols),
])
# (B) Categorical NB
prep_cat = Pipeline([
    ("cols", ColumnTransformer([
        ("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                          ("bins", KBinsDiscretizer(n_bins=5, encode="ordinal", strategy="quantile"))]), num_cols),
        ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")), ("ord", OrdinalEncoder())]), cat_cols),
    ])),
    ("to_int", FunctionTransformer(to_int)),
])
print("\nNumeric:", num_cols, "\nCategorical:", cat_cols)
print("Train:", X_train.shape, "Test:", X_test.shape)

## Step 4: Exploratory Data Analysis (EDA)

In [ ]:
print(df["Loan_Status"].value_counts(normalize=True).round(3))
print(df.describe().T.round(2))

fig, axes = plt.subplots(2, 3, figsize=(18, 9))
sns.countplot(data=df, x="Loan_Status", ax=axes[0, 0]); axes[0, 0].set_title("Approved vs rejected")
sns.boxplot(data=df, x="Loan_Status", y="Credit_Score", ax=axes[0, 1]); axes[0, 1].set_title("Credit score")
sns.boxplot(data=df, x="Loan_Status", y="Monthly_Income", ax=axes[0, 2]); axes[0, 2].set_yscale("log"); axes[0, 2].set_title("Monthly income (log)")
sns.countplot(data=df, x="Education", hue="Loan_Status", ax=axes[1, 0]); axes[1, 0].set_title("Education")
sns.countplot(data=df, x="Employment", hue="Loan_Status", ax=axes[1, 1]); axes[1, 1].set_title("Employment")
sns.countplot(data=df, x="Property_Area", hue="Loan_Status", ax=axes[1, 2]); axes[1, 2].set_title("Property area")
plt.tight_layout(); plt.show()

print("\nApproval rate by category:")
for c in ["Education", "Married", "Employment", "Property_Area"]:
    print(df.groupby(c)["Loan_Status"].apply(lambda s: (s == "Approved").mean()).round(3).to_dict())

## Step 5: Model Training
Train Gaussian NB and Categorical NB, and tune the **Laplace smoothing** parameter `alpha` for the categorical model with grid search.

In [ ]:
from sklearn.naive_bayes import GaussianNB, CategoricalNB
from sklearn.model_selection import GridSearchCV, StratifiedKFold

gauss = Pipeline([("prep", prep_gauss), ("nb", GaussianNB())]).fit(X_train, y_train)

cat_nb = Pipeline([("prep", prep_cat), ("nb", CategoricalNB(min_categories=10))])
search = GridSearchCV(cat_nb, {"nb__alpha": [0.01, 0.1, 0.5, 1.0, 2.0, 5.0]},
                      cv=StratifiedKFold(5, shuffle=True, random_state=42), scoring="roc_auc").fit(X_train, y_train)
cat_best = search.best_estimator_
print("Best Laplace alpha for Categorical NB:", search.best_params_["nb__alpha"], "| CV ROC-AUC:", round(search.best_score_, 4))

## Step 6: Model Evaluation
Compare both Naive Bayes variants on the test set with accuracy, precision, recall, F1, ROC-AUC, confusion matrices and a **decision-threshold analysis** (banks care more about avoiding bad approvals).

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, roc_curve, classification_report

rows, probs = [], {}
for name, model in {"Gaussian NB": gauss, "Categorical NB": cat_best}.items():
    pred = model.predict(X_test); prob = model.predict_proba(X_test)[:, 1]; probs[name] = prob
    rows.append({"Model": name, "Accuracy": accuracy_score(y_test, pred), "Precision": precision_score(y_test, pred),
                 "Recall": recall_score(y_test, pred), "F1": f1_score(y_test, pred), "ROC-AUC": roc_auc_score(y_test, prob)})
print(pd.DataFrame(rows).set_index("Model").round(4))
print("\nBaseline (always predict 'Approved'):", round(y_test.mean(), 4))
print("\nGaussian NB report:\n", classification_report(y_test, gauss.predict(X_test), target_names=["Rejected", "Approved"]))

fig, axes = plt.subplots(1, 3, figsize=(18, 4.8))
for ax, (name, model) in zip(axes[:2], {"Gaussian NB": gauss, "Categorical NB": cat_best}.items()):
    sns.heatmap(confusion_matrix(y_test, model.predict(X_test)), annot=True, fmt="d", cmap="Blues", ax=ax,
                xticklabels=["Rejected", "Approved"], yticklabels=["Rejected", "Approved"])
    ax.set_title(name); ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
for name, prob in probs.items():
    fpr, tpr, _ = roc_curve(y_test, prob); axes[2].plot(fpr, tpr, label=f"{name} (AUC={roc_auc_score(y_test, prob):.3f})")
axes[2].plot([0, 1], [0, 1], "k--"); axes[2].legend(); axes[2].set_title("ROC curves"); axes[2].set_xlabel("FPR"); axes[2].set_ylabel("TPR")
plt.tight_layout(); plt.show()

In [ ]:
# Stricter approval threshold -> fewer risky approvals (higher precision) but more good applicants rejected (lower recall)
prob = probs["Gaussian NB"]
table = []
for t in [0.3, 0.5, 0.7, 0.8, 0.9]:
    pred = (prob >= t).astype(int)
    table.append({"Approve if P >=": t, "Precision": precision_score(y_test, pred), "Recall": recall_score(y_test, pred), "Approved share": pred.mean()})
print(pd.DataFrame(table).round(3).to_string(index=False))

## Step 7: Prediction on New Data

In [ ]:
new_applicants = pd.DataFrame([
    {"Age": 35, "Monthly_Income": 90000, "Loan_Amount_K": 900, "Loan_Term_Months": 360, "Credit_Score": 760, "Employment": "Salaried",
     "Education": "Graduate", "Married": "Yes", "Dependents": 1, "Property_Area": "Urban", "Existing_EMI": 9000},
    {"Age": 28, "Monthly_Income": 20000, "Loan_Amount_K": 900, "Loan_Term_Months": 180, "Credit_Score": 520, "Employment": "Unemployed",
     "Education": "Not Graduate", "Married": "No", "Dependents": 3, "Property_Area": "Rural", "Existing_EMI": 5000},
])
p = gauss.predict_proba(new_applicants)[:, 1]
for i, pa in enumerate(p, 1):
    print(f"Applicant {i}: P(approved) = {pa:.3f} -> {'Approve' if pa >= 0.5 else 'Reject'}")

## Conclusion
Both Naive Bayes variants clearly beat the always-approve baseline (accuracy about 74-75% and ROC-AUC about 0.80-0.81, versus 59% for always approving), with credit score, income-to-loan ratio and employment status driving the decision. Categorical NB (discretised features with Laplace smoothing) came out slightly ahead of Gaussian NB (one-hot categoricals); the tuned `alpha` mostly matters for rare categories. Naive Bayes probabilities tend to be over-confident (many predictions are close to 0 or 1), so use them for ranking and thresholding rather than as exact probabilities. Raising the approval threshold trades recall for precision, which lets a bank control default risk. Naive Bayes is a fast, transparent baseline; logistic regression or gradient boosting would be natural next steps.